# 6.20 — Learning Rate Schedules

A learning-rate schedule chooses the step size $\eta_t$ used by gradient descent at each training step. In this lesson, you will build constant, warmup, step-decay, cosine-decay, and one-cycle schedules from scratch in NumPy, then inspect how each schedule changes the actual parameter movement on a tiny optimization problem.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build learning-rate schedules one idea at a time. Run each cell in order and read the printed intermediate values — every schedule is just arithmetic on the step index, and every update is shown. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, schedule formulas, and tiny optimization loops.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any randomized toy data.

### 1. The learning rate is the size of a gradient step

Gradient descent moves a parameter by $\theta_{t+1}=\theta_t-\eta_t g_t$. The gradient $g_t$ points uphill for the loss, so the minus sign moves downhill. The learning rate $\eta_t$ is not the direction; it is the distance multiplier. A schedule matters because the same gradient can produce a cautious nudge or an overshooting jump depending only on $\eta_t$.

In [ ]:
theta_w = 2.0                         # one scalar parameter before the update.
grad_w = 2.1                          # current derivative dL/dtheta.
eta_w = 0.05                          # learning rate for this step.
move_w = eta_w * grad_w               # amount subtracted from theta.
theta_next_w = theta_w - move_w        # gradient descent update.
print("move:", round(move_w, 3))
print("theta next:", round(theta_next_w, 3))
assert round(move_w, 3) == 0.105
assert round(theta_next_w, 3) == 1.895

▶ What you'll see: the gradient says which way to move, and the learning rate scales that move to 0.105.

In [ ]:
eta_grid_w = np.array([0.01, 0.05, 0.20])        # compare small, moderate, and large rates.
next_grid_w = theta_w - eta_grid_w * grad_w      # same gradient, different step sizes.
print("next theta values:", np.round(next_grid_w, 3))
plt.figure(figsize=(4.4, 3))
plt.bar(["0.01", "0.05", "0.20"], theta_w - next_grid_w, color="teal")
plt.title("1: same gradient, different movement")
plt.xlabel("learning rate η")
plt.ylabel("subtracted amount ηg")
plt.show()

▶ What you'll see: movement grows linearly with the learning rate; the direction is unchanged.

*Why it's done this way:* the derivative is measured in loss-change per parameter-change, so multiplying by $\eta_t$ converts slope information into an actual parameter displacement. Scheduling $\eta_t$ is therefore scheduling how much trust we place in each noisy gradient estimate.

### 2. Warmup: start small while signals stabilize

Warmup increases the rate gradually from a tiny value to a target maximum. Early gradients can be large or poorly calibrated because weights, activations, and optimizer statistics are still settling. A linear warmup says: spend the first few steps learning the scale of the problem before taking full-size steps.

In [ ]:
steps_w = np.arange(12)                         # training step indices.
warmup_steps_w = 5                              # number of steps spent ramping up.
eta_max_w = 0.10                                # target learning rate after warmup.
warmup_w = eta_max_w * np.minimum(1.0, (steps_w + 1) / warmup_steps_w)
print("warmup rates:", np.round(warmup_w, 3))
assert np.allclose(np.round(warmup_w[:5], 3), [0.02, 0.04, 0.06, 0.08, 0.10])

▶ What you'll see: the schedule climbs 0.02, 0.04, 0.06, 0.08, 0.10, then stays at the target.

In [ ]:
grad0_w = 3.0                                  # pretend the early gradient is large.
updates_w = warmup_w[:6] * grad0_w             # actual movement caused by the schedule.
print("first six update sizes:", np.round(updates_w, 3))
plt.figure(figsize=(4.6, 3))
plt.plot(steps_w, warmup_w, marker="o", color="seagreen")
plt.title("2: linear warmup schedule")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: the plotted ramp prevents the earliest updates from being as large as later updates.

*Why it's done this way:* a linear ramp is the simplest interpolation between “almost do not move” and “use the intended rate.” The formula $\eta_t=\eta_{max}(t+1)/T_w$ keeps the increase predictable, so every early update is a controlled fraction of the full update.

### 3. Step decay: drop the rate when progress plateaus

Step decay keeps a rate constant for a while, then multiplies it by a factor such as 0.1 at chosen milestones. The idea is coarse but practical: use decisive movement early, then reduce the step size so the optimizer can settle into a narrower part of the loss surface.

In [ ]:
steps3_w = np.arange(16)                         # training steps to display.
eta0_w = 0.20                                    # initial learning rate.
drop_every_w = 5                                 # milestone interval.
gamma_w = 0.5                                    # multiplicative drop factor.
step_decay_w = eta0_w * gamma_w ** (steps3_w // drop_every_w)
print("step-decay rates:", np.round(step_decay_w, 3))
assert np.allclose(np.round(step_decay_w[[0, 5, 10, 15]], 3), [0.20, 0.10, 0.05, 0.025])

▶ What you'll see: the rate stays flat, drops at step 5, drops again at step 10, and drops again at step 15.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.step(steps3_w, step_decay_w, where="post", color="darkorange")
plt.scatter(steps3_w, step_decay_w, color="darkorange")
plt.title("3: step decay spends rate in plateaus")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a staircase shape; the discontinuities are intentional milestone decisions.

*Why it's done this way:* multiplying by $\gamma$ changes the update scale without changing the gradient formula. Large plateaus help escape broad high-loss regions; later smaller plateaus make oscillation around a minimum less likely.

### 4. Cosine decay: settle smoothly instead of jumping

Cosine decay lowers the rate continuously from $\eta_{max}$ to $\eta_{min}$:
$$\eta_t=\eta_{min}+\frac12(\eta_{max}-\eta_{min})(1+\cos(\pi t/T)).$$
The cosine starts with a gentle slope, falls fastest in the middle, and ends gently. That shape avoids sudden changes while still spending most of training moving from bold steps to careful steps.

In [ ]:
T_w = 20                                      # total schedule length.
t_cos_w = np.arange(T_w + 1)                  # include both endpoints 0 and T.
eta_min_w = 0.01                              # final floor.
eta_max2_w = 0.10                             # initial ceiling.
cosine_w = eta_min_w + 0.5 * (eta_max2_w - eta_min_w) * (1 + np.cos(np.pi * t_cos_w / T_w))
print("start/middle/end:", np.round([cosine_w[0], cosine_w[10], cosine_w[-1]], 3))
assert np.allclose(np.round([cosine_w[0], cosine_w[10], cosine_w[-1]], 3), [0.10, 0.055, 0.01])

▶ What you'll see: the schedule begins at 0.100, reaches the midpoint 0.055 halfway through, and ends at 0.010.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(t_cos_w, cosine_w, marker="o", color="purple")
plt.title("4: cosine decay from η_max to η_min")
plt.xlabel("step t")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a smooth falling curve with no sharp milestone jumps.

*Why it's done this way:* the term $(1+\cos(\pi t/T))/2$ is exactly 1 at $t=0$ and 0 at $t=T$, so it acts like a smooth interpolation weight between maximum and minimum learning rates.

### 5. One-cycle: explore upward, then anneal downward

A one-cycle schedule first increases the learning rate, then decreases it to a very small value. The rising half can help the optimizer move out of narrow or poorly conditioned regions; the falling half then anneals the updates so the final parameters settle. We can build it by concatenating two linear pieces.

In [ ]:
up_steps_w = 5                                  # length of the rising phase.
down_steps_w = 10                               # length of the falling phase.
eta_low_w = 0.02                                # beginning rate.
eta_peak_w = 0.12                               # exploratory peak.
eta_final_w = 0.005                             # final annealed rate.
up_w = np.linspace(eta_low_w, eta_peak_w, up_steps_w, endpoint=False)
down_w = np.linspace(eta_peak_w, eta_final_w, down_steps_w)
one_cycle_w = np.concatenate([up_w, down_w])
print("one-cycle endpoints:", np.round([one_cycle_w[0], one_cycle_w[4], one_cycle_w[5], one_cycle_w[-1]], 3))
assert np.allclose(np.round([one_cycle_w[0], one_cycle_w[5], one_cycle_w[-1]], 3), [0.02, 0.12, 0.005])

▶ What you'll see: the schedule starts low, reaches a peak, then finishes below where it started.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(np.arange(len(one_cycle_w)), one_cycle_w, marker="o", color="crimson")
plt.title("5: one-cycle learning rate")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: an up-then-down triangle-like curve, with the longest phase spent annealing.

*Why it's done this way:* one-cycle separates “search” from “settle.” The peak deliberately allows larger movement after warmup, while the final tiny rate reduces the risk that noisy minibatch gradients keep shaking the solution.

### 6. Schedules change optimization trajectories

To see schedules as training behavior rather than just curves, optimize $L(\theta)=(\theta-3)^2$. Its gradient is $2(\theta-3)$. A larger learning rate moves faster toward 3, but if it is too large it can cross the minimum repeatedly. Scheduled rates let us move aggressively early and carefully later.

In [ ]:
def grad_loss_w(theta):                         # derivative of (theta - 3)^2.
    return 2 * (theta - 3.0)

n_steps6_w = 30
constant_sched_w = np.full(n_steps6_w, 0.08)
cos_sched_w = 0.01 + 0.5 * (0.16 - 0.01) * (1 + np.cos(np.pi * np.arange(n_steps6_w) / (n_steps6_w - 1)))
print("constant first/last:", constant_sched_w[0], constant_sched_w[-1])
print("cosine first/last:", round(cos_sched_w[0], 3), round(cos_sched_w[-1], 3))
assert round(cos_sched_w[0], 3) == 0.16 and round(cos_sched_w[-1], 3) == 0.01

▶ What you'll see: the cosine schedule begins twice as high as the constant rate and ends much lower.

In [ ]:
def run_schedule_w(schedule):
    theta_hist = [0.0]
    for eta in schedule:
        theta_hist.append(theta_hist[-1] - eta * grad_loss_w(theta_hist[-1]))
    return np.array(theta_hist)

theta_const_w = run_schedule_w(constant_sched_w)
theta_cos_w = run_schedule_w(cos_sched_w)
print("final theta constant/cosine:", round(theta_const_w[-1], 3), round(theta_cos_w[-1], 3))
assert abs(theta_const_w[-1] - 3.0) < 0.02
assert abs(theta_cos_w[-1] - 3.0) < 0.02

▶ What you'll see: both trajectories end near the minimum, but they get there with different step sizes over time.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(theta_const_w, label="constant η=0.08", color="gray")
plt.plot(theta_cos_w, label="cosine 0.16→0.01", color="purple")
plt.axhline(3.0, color="black", linestyle="--", label="minimum θ=3")
plt.title("6: schedules create different paths")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the cosine schedule moves quickly at first, then flattens as the rate decays.

*Why it's done this way:* optimization is repeated local approximation. Early in training we are far from the solution, so larger steps are useful; near the solution, smaller steps reduce bouncing around the minimum.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Apply one gradient step

**Goal.** Use $\theta_{t+1}=\theta_t-\eta g$ once, because every schedule ultimately changes this one multiplication.

In [ ]:
theta_b1 = 2.0
grad_b1 = 2.1
eta_b1 = 0.05
next_b1 = theta_b1 - eta_b1 * grad_b1
print("next theta:", round(next_b1, 3))
assert round(next_b1, 3) == 1.895

In [ ]:
center_b1 = theta_b1 - grad_b1 / 2
grid_b1 = np.linspace(1.6, 2.2, 100)
loss_b1 = (grid_b1 - center_b1) ** 2
plt.figure(figsize=(4, 3))
plt.plot(grid_b1, loss_b1, color="slateblue")
plt.scatter([theta_b1, next_b1], [(theta_b1 - center_b1) ** 2, (next_b1 - center_b1) ** 2],
            color=["crimson", "seagreen"], zorder=3)
plt.annotate("before", (theta_b1, (theta_b1 - center_b1) ** 2), xytext=(5, 5), textcoords="offset points")
plt.annotate("after", (next_b1, (next_b1 - center_b1) ** 2), xytext=(5, -12), textcoords="offset points")
plt.title("Basic 1: one step on a 1D loss")
plt.xlabel("theta")
plt.ylabel("toy loss")
plt.show()

▶ What you'll see: the single update moves theta leftward and slightly down the toy quadratic.

▶ What you'll see: the parameter moves from 2.000 to 1.895.

👀 Takeaway: the learning rate scales the gradient before subtraction.

### Basic 2 — Compare update sizes

**Goal.** Keep the gradient fixed and vary the learning rate, because this isolates what the schedule controls.

In [ ]:
grad_b2 = 2.0
etas_b2 = np.array([0.01, 0.05, 0.10, 0.20])
updates_b2 = etas_b2 * grad_b2
print("updates:", np.round(updates_b2, 3))
assert np.allclose(updates_b2, [0.02, 0.10, 0.20, 0.40])
plt.figure(figsize=(4, 3))
plt.bar([str(x) for x in etas_b2], updates_b2, color="teal")
plt.title("Basic 2: update = ηg")
plt.xlabel("η")
plt.ylabel("update size")
plt.show()

▶ What you'll see: doubling the learning rate doubles the update.

👀 Takeaway: schedules are multiplicative controls on step length.

### Basic 3 — Build a constant schedule

**Goal.** Make an array of identical learning rates, because constant-rate training is the baseline schedule.

In [ ]:
steps_b3 = np.arange(8)
eta_b3 = 0.05
sched_b3 = np.full_like(steps_b3, eta_b3, dtype=float)
print("constant schedule:", sched_b3)
assert np.allclose(sched_b3, 0.05)
plt.figure(figsize=(4, 3))
plt.plot(steps_b3, sched_b3, marker="o", color="gray")
plt.ylim(0, 0.08)
plt.title("Basic 3: constant η")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a flat line at 0.05.

👀 Takeaway: a constant schedule trusts every step equally.

### Basic 4 — Build linear warmup

**Goal.** Ramp from zero toward a target rate, because early updates are often the most fragile.

In [ ]:
steps_b4 = np.arange(6)
warmup_steps_b4 = 5
eta_max_b4 = 0.10
sched_b4 = eta_max_b4 * np.minimum(1.0, (steps_b4 + 1) / warmup_steps_b4)
print("warmup:", np.round(sched_b4, 3))
assert np.allclose(np.round(sched_b4, 3), [0.02, 0.04, 0.06, 0.08, 0.10, 0.10])
plt.figure(figsize=(4, 3))
plt.plot(steps_b4, sched_b4, marker="o", color="seagreen")
plt.title("Basic 4: linear warmup")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a straight ramp that reaches 0.10 and then stays there.

👀 Takeaway: warmup makes early movement a fraction of the full learning rate.

### Basic 5 — Build step decay

**Goal.** Drop the learning rate at fixed milestones, because smaller late steps help settling.

In [ ]:
steps_b5 = np.arange(12)
eta0_b5 = 0.20
gamma_b5 = 0.5
drop_every_b5 = 4
sched_b5 = eta0_b5 * gamma_b5 ** (steps_b5 // drop_every_b5)
print("step decay:", np.round(sched_b5, 3))
assert np.allclose(np.round(sched_b5[[0, 4, 8]], 3), [0.20, 0.10, 0.05])
plt.figure(figsize=(4, 3))
plt.step(steps_b5, sched_b5, where="post", color="darkorange")
plt.title("Basic 5: milestone drops")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a staircase from 0.20 to 0.10 to 0.05.

👀 Takeaway: step decay changes the trust level abruptly at chosen milestones.

### Basic 6 — Build cosine decay

**Goal.** Compute the core cosine formula, because it is a smooth high-to-low schedule.

In [ ]:
T_b6 = 10
t_b6 = np.arange(T_b6 + 1)
eta_min_b6 = 0.01
eta_max_b6 = 0.10
sched_b6 = eta_min_b6 + 0.5 * (eta_max_b6 - eta_min_b6) * (1 + np.cos(np.pi * t_b6 / T_b6))
print("cosine endpoints:", round(sched_b6[0], 3), round(sched_b6[-1], 3))
assert round(sched_b6[0], 3) == 0.10 and round(sched_b6[-1], 3) == 0.01
plt.figure(figsize=(4, 3))
plt.plot(t_b6, sched_b6, marker="o", color="purple")
plt.title("Basic 6: cosine decay")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: the curve starts at the maximum and lands exactly on the minimum.

👀 Takeaway: cosine decay is smooth interpolation from bold to careful updates.

### Basic 7 — Clip a schedule floor

**Goal.** Keep the rate from going below a floor, because some training runs should continue learning slowly instead of stopping.

In [ ]:
raw_b7 = np.linspace(0.10, -0.02, 7)
floor_b7 = 0.01
clipped_b7 = np.maximum(raw_b7, floor_b7)
print("raw:", np.round(raw_b7, 3))
print("floored:", np.round(clipped_b7, 3))
assert clipped_b7[-1] == 0.01
plt.figure(figsize=(4, 3))
plt.plot(raw_b7, marker="o", label="raw")
plt.plot(clipped_b7, marker="s", label="floored")
plt.title("Basic 7: learning-rate floor")
plt.legend()
plt.show()

▶ What you'll see: the floored schedule stops descending once it reaches 0.01.

👀 Takeaway: a floor preserves tiny but nonzero movement.

### Basic 8 — Compute cumulative learning-rate budget

**Goal.** Sum rates across steps, because total scheduled movement depends on both rate height and duration.

In [ ]:
sched_b8 = np.array([0.10, 0.10, 0.05, 0.05, 0.01])
budget_b8 = np.cumsum(sched_b8)
print("cumulative budget:", np.round(budget_b8, 3))
assert round(budget_b8[-1], 3) == 0.31
plt.figure(figsize=(4, 3))
plt.plot(budget_b8, marker="o", color="navy")
plt.title("Basic 8: cumulative η budget")
plt.xlabel("step")
plt.ylabel("sum of η so far")
plt.show()

▶ What you'll see: the budget rises fastest during high-rate steps.

👀 Takeaway: schedules allocate a finite amount of movement over time.

### Basic 9 — Apply a schedule to fixed gradients

**Goal.** Convert a rate schedule into actual update magnitudes, because the schedule affects parameters only through updates.

In [ ]:
grads_b9 = np.array([3.0, 2.0, 1.0, 0.5])
sched_b9 = np.array([0.02, 0.04, 0.08, 0.08])
updates_b9 = sched_b9 * grads_b9
print("updates:", np.round(updates_b9, 3))
assert np.allclose(updates_b9, [0.06, 0.08, 0.08, 0.04])
plt.figure(figsize=(4, 3))
plt.bar(np.arange(4), updates_b9, color="crimson")
plt.title("Basic 9: η_t times gradient")
plt.xlabel("step")
plt.ylabel("update magnitude")
plt.show()

▶ What you'll see: a larger rate can offset a smaller gradient, so update size is their product.

👀 Takeaway: the schedule and gradient jointly determine movement.

### Basic 10 — Plot schedule and loss together

**Goal.** Put a learning-rate curve beside a toy loss curve, because the schedule is meaningful only through training progress.

In [ ]:
steps_b10 = np.arange(20)
sched_b10 = 0.01 + 0.5 * (0.10 - 0.01) * (1 + np.cos(np.pi * steps_b10 / 19))
loss_b10 = np.exp(-0.18 * steps_b10) + 0.03 * sched_b10 / sched_b10.max()
print("first/last loss:", round(loss_b10[0], 3), round(loss_b10[-1], 3))
assert loss_b10[-1] < loss_b10[0]
plt.figure(figsize=(5, 3))
plt.plot(steps_b10, sched_b10 / sched_b10.max(), label="scaled η", color="purple")
plt.plot(steps_b10, loss_b10 / loss_b10.max(), label="scaled loss", color="teal")
plt.title("Basic 10: rate and progress")
plt.xlabel("step")
plt.legend()
plt.show()

▶ What you'll see: the rate decays while the toy loss decreases.

👀 Takeaway: schedules are judged by whether they help the loss decrease reliably.

## 🟡 Easy

### Easy 1 — Train a quadratic with constant and decayed rates

**Goal.** Optimize $L(\theta)=(\theta-3)^2$ with two schedules, because a schedule changes the trajectory even on a one-parameter problem.

In [ ]:
steps_e1 = 35
eta_const_e1 = np.full(steps_e1, 0.08)
eta_decay_e1 = 0.01 + 0.5 * (0.16 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_e1) / (steps_e1 - 1)))
print("decay start/end:", round(eta_decay_e1[0], 3), round(eta_decay_e1[-1], 3))
assert round(eta_decay_e1[0], 3) == 0.16 and round(eta_decay_e1[-1], 3) == 0.01

▶ What you'll see: the cosine-decayed schedule starts high and ends low.

In [ ]:
theta_const_e1 = [0.0]
theta_decay_e1 = [0.0]
for i_e1 in range(steps_e1):
    theta_const_e1.append(theta_const_e1[-1] - eta_const_e1[i_e1] * 2 * (theta_const_e1[-1] - 3.0))
    theta_decay_e1.append(theta_decay_e1[-1] - eta_decay_e1[i_e1] * 2 * (theta_decay_e1[-1] - 3.0))
theta_const_e1 = np.array(theta_const_e1)
theta_decay_e1 = np.array(theta_decay_e1)
print("final theta:", round(theta_const_e1[-1], 3), round(theta_decay_e1[-1], 3))
assert abs(theta_const_e1[-1] - 3.0) < 0.01 and abs(theta_decay_e1[-1] - 3.0) < 0.01

▶ What you'll see: both schedules approach the optimum.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(theta_const_e1, label="constant", color="gray")
plt.plot(theta_decay_e1, label="cosine decay", color="purple")
plt.axhline(3.0, color="black", linestyle="--")
plt.title("Easy 1: optimization trajectory")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the decayed schedule moves quickly early and then flattens near 3.

👀 Takeaway: schedules turn the same gradient rule into different training paths.

### Easy 2 — Combine warmup and cosine decay

**Goal.** Use warmup first and cosine decay afterward, because many deep-learning runs need both stable starts and smooth settling.

In [ ]:
warm_steps_e2 = 5
decay_steps_e2 = 25
eta_peak_e2 = 0.12
eta_min_e2 = 0.01
warm_e2 = eta_peak_e2 * (np.arange(1, warm_steps_e2 + 1) / warm_steps_e2)
t_e2 = np.arange(decay_steps_e2)
decay_e2 = eta_min_e2 + 0.5 * (eta_peak_e2 - eta_min_e2) * (1 + np.cos(np.pi * t_e2 / (decay_steps_e2 - 1)))
sched_e2 = np.concatenate([warm_e2, decay_e2[1:]])
print("length:", len(sched_e2), "peak:", round(sched_e2.max(), 3), "final:", round(sched_e2[-1], 3))
assert round(sched_e2.max(), 3) == 0.12 and round(sched_e2[-1], 3) == 0.01

▶ What you'll see: the schedule reaches 0.12 after warmup and ends at 0.01.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(sched_e2, marker="o", color="seagreen")
plt.axvline(warm_steps_e2 - 1, color="gray", linestyle="--", label="warmup ends")
plt.title("Easy 2: warmup + cosine")
plt.xlabel("step")
plt.ylabel("η_t")
plt.legend()
plt.show()

▶ What you'll see: an initial ramp followed by a smooth decay.

👀 Takeaway: warmup and cosine solve different parts of the training timeline.

### Easy 3 — Compare schedule budgets

**Goal.** Compare total learning-rate mass, because two schedules with the same maximum can spend very different amounts of movement.

In [ ]:
steps_e3 = 30
const_e3 = np.full(steps_e3, 0.06)
step_e3 = 0.12 * 0.5 ** (np.arange(steps_e3) // 10)
cos_e3 = 0.01 + 0.5 * (0.12 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_e3) / (steps_e3 - 1)))
budgets_e3 = np.array([const_e3.sum(), step_e3.sum(), cos_e3.sum()])
print("budgets:", np.round(budgets_e3, 3))
assert np.all(budgets_e3 > 0)

▶ What you'll see: each schedule spends a different cumulative learning-rate budget.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["constant", "step", "cosine"], budgets_e3, color=["gray", "orange", "purple"])
plt.title("Easy 3: total η budget")
plt.ylabel("sum_t η_t")
plt.show()

▶ What you'll see: the highest budget belongs to the schedule that keeps rates larger for longer.

👀 Takeaway: matching peak learning rates does not match total movement.

### Easy 4 — Detect overshooting on a quadratic

**Goal.** Show that a too-large learning rate can bounce across the minimum, because update size can exceed the useful local distance.

In [ ]:
eta_good_e4 = 0.20
eta_bad_e4 = 1.10
steps_e4 = 10
theta_good_e4 = [0.0]
theta_bad_e4 = [0.0]
for step_e4 in range(steps_e4):
    theta_good_e4.append(theta_good_e4[-1] - eta_good_e4 * 2 * (theta_good_e4[-1] - 3.0))
    theta_bad_e4.append(theta_bad_e4[-1] - eta_bad_e4 * 2 * (theta_bad_e4[-1] - 3.0))
theta_good_e4 = np.array(theta_good_e4)
theta_bad_e4 = np.array(theta_bad_e4)
print("bad trajectory first values:", np.round(theta_bad_e4[:5], 3))
assert np.any(theta_bad_e4 > 3.0) and np.any(theta_bad_e4 < 0.0)

▶ What you'll see: the bad trajectory alternates around the optimum with growing magnitude.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(theta_good_e4, marker="o", label="η=0.20", color="teal")
plt.plot(theta_bad_e4, marker="x", label="η=1.10", color="crimson")
plt.axhline(3.0, color="black", linestyle="--")
plt.title("Easy 4: overshooting")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the high-rate path crosses the target repeatedly instead of settling.

👀 Takeaway: schedules must respect the curvature of the loss surface.

### Easy 5 — Mini-batch noise and late small rates

**Goal.** Optimize with noisy gradients, because minibatches estimate the true gradient rather than measuring it exactly.

In [ ]:
steps_e5 = 60
rng_e5 = np.random.default_rng(5)
noise_e5 = rng_e5.normal(0, 0.5, size=steps_e5)
const_e5 = np.full(steps_e5, 0.08)
decay_e5 = 0.01 + 0.5 * (0.14 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_e5) / (steps_e5 - 1)))
print("noise mean/std:", round(float(noise_e5.mean()), 3), round(float(noise_e5.std()), 3))
assert abs(noise_e5.mean()) < 0.2

▶ What you'll see: the synthetic minibatch noise is centered near zero.

In [ ]:
def noisy_run_e5(schedule_e5):
    theta_e5 = [0.0]
    for i_e5, eta_e5 in enumerate(schedule_e5):
        noisy_grad_e5 = 2 * (theta_e5[-1] - 3.0) + noise_e5[i_e5]
        theta_e5.append(theta_e5[-1] - eta_e5 * noisy_grad_e5)
    return np.array(theta_e5)

path_const_e5 = noisy_run_e5(const_e5)
path_decay_e5 = noisy_run_e5(decay_e5)
print("final errors:", round(abs(path_const_e5[-1] - 3), 3), round(abs(path_decay_e5[-1] - 3), 3))
assert abs(path_decay_e5[-1] - 3) < 0.2

▶ What you'll see: both runs are near the optimum, but late movement differs.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(path_const_e5, label="constant", color="gray")
plt.plot(path_decay_e5, label="cosine decay", color="purple")
plt.axhline(3.0, color="black", linestyle="--")
plt.title("Easy 5: noisy gradients")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the decayed path jitters less late because its final rates are smaller.

👀 Takeaway: decaying rates reduce the effect of noisy gradient estimates near convergence.

## 🔴 Advanced

### Advanced 1 — Sweep maximum learning rate

**Goal.** Try several peak rates with the same cosine shape, because the schedule formula still needs a scale that matches the problem.

In [ ]:
peaks_a1 = np.array([0.04, 0.10, 0.30, 0.80])
steps_a1 = 45
final_losses_a1 = []
for peak_a1 in peaks_a1:
    sched_a1 = 0.005 + 0.5 * (peak_a1 - 0.005) * (1 + np.cos(np.pi * np.arange(steps_a1) / (steps_a1 - 1)))
    theta_a1 = 0.0
    for eta_a1 in sched_a1:
        theta_a1 = theta_a1 - eta_a1 * 2 * (theta_a1 - 3.0)
        theta_a1 = float(np.clip(theta_a1, -50, 50))
    final_losses_a1.append((theta_a1 - 3.0) ** 2)
final_losses_a1 = np.array(final_losses_a1)
print("final losses:", np.round(final_losses_a1, 5))
assert final_losses_a1[1] < final_losses_a1[0]

▶ What you'll see: too-low peaks can learn slowly, while reasonable peaks finish closer to the optimum.

In [ ]:
best_peak_a1 = peaks_a1[int(np.argmin(final_losses_a1))]
plt.figure(figsize=(5, 3))
plt.plot(peaks_a1, final_losses_a1, marker="o", color="navy")
plt.axvline(best_peak_a1, color="crimson", linestyle="--", label=f"best peak={best_peak_a1:.2f}")
plt.title("Advanced 1: peak-rate sweep")
plt.xlabel("η_max")
plt.ylabel("final loss")
plt.legend()
plt.show()

▶ What you'll see: one peak gives the smallest final loss for this toy problem.

👀 Takeaway: schedule shape and schedule scale are separate hyperparameters.

### Advanced 2 — One-cycle with momentum in the opposite direction

**Goal.** Pair a rising learning rate with falling momentum, because one-cycle training often trades stability from momentum against exploration from larger steps.

In [ ]:
up_a2 = np.linspace(0.02, 0.16, 8, endpoint=False)
down_a2 = np.linspace(0.16, 0.005, 22)
lr_a2 = np.concatenate([up_a2, down_a2])
mom_a2 = np.linspace(0.95, 0.85, len(up_a2)).tolist() + np.linspace(0.85, 0.95, len(down_a2)).tolist()
mom_a2 = np.array(mom_a2)
print("lr first/peak/final:", round(lr_a2[0], 3), round(lr_a2.max(), 3), round(lr_a2[-1], 3))
print("momentum first/min/final:", round(mom_a2[0], 3), round(mom_a2.min(), 3), round(mom_a2[-1], 3))
assert round(lr_a2.max(), 3) == 0.16 and round(mom_a2.min(), 3) == 0.85

▶ What you'll see: learning rate rises while momentum falls, then learning rate falls while momentum rises.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lr_a2 / lr_a2.max(), label="scaled learning rate", color="crimson")
plt.plot((mom_a2 - mom_a2.min()) / (mom_a2.max() - mom_a2.min()), label="scaled momentum", color="teal")
plt.title("Advanced 2: one-cycle LR and momentum")
plt.xlabel("step")
plt.legend()
plt.show()

▶ What you'll see: the two curves move in opposite directions during the cycle.

👀 Takeaway: one-cycle can schedule both how far to step and how much velocity to carry.

### Advanced 3 — Simulate restarts with cosine cycles

**Goal.** Restart cosine decay repeatedly, because restarts periodically reintroduce larger exploratory steps.

In [ ]:
cycle_lengths_a3 = [8, 12, 16]
eta_max_a3 = 0.12
eta_min_a3 = 0.01
sched_parts_a3 = []
for length_a3 in cycle_lengths_a3:
    t_a3 = np.arange(length_a3)
    part_a3 = eta_min_a3 + 0.5 * (eta_max_a3 - eta_min_a3) * (1 + np.cos(np.pi * t_a3 / (length_a3 - 1)))
    sched_parts_a3.append(part_a3)
sched_a3 = np.concatenate(sched_parts_a3)
print("restart positions:", np.cumsum(cycle_lengths_a3)[:-1])
assert len(sched_a3) == sum(cycle_lengths_a3)

▶ What you'll see: the schedule is built from three cosine cycles.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(sched_a3, color="purple")
for pos_a3 in np.cumsum(cycle_lengths_a3)[:-1]:
    plt.axvline(pos_a3, color="gray", linestyle="--")
plt.title("Advanced 3: cosine restarts")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: each restart jumps back to a high learning rate before decaying again.

👀 Takeaway: restarts deliberately alternate settling phases with renewed exploration.

### Advanced 4 — Schedule weight decay strength through update size

**Goal.** Inspect how learning-rate decay also shrinks the effective L2 weight-decay update, because regularization steps are multiplied by $\eta_t$.

In [ ]:
weights_a4 = np.array([2.0, -1.0, 0.5])
lam_a4 = 0.1
sched_a4 = np.array([0.10, 0.05, 0.01])
decay_updates_a4 = np.array([eta_a4 * lam_a4 * weights_a4 for eta_a4 in sched_a4])
print("decay updates:\n", np.round(decay_updates_a4, 3))
assert np.allclose(decay_updates_a4[0], [0.02, -0.01, 0.005])

▶ What you'll see: the same weight vector receives smaller shrinkage as the learning rate decays.

In [ ]:
norms_a4 = np.linalg.norm(decay_updates_a4, axis=1)
plt.figure(figsize=(4.5, 3))
plt.bar(["η=.10", "η=.05", "η=.01"], norms_a4, color="darkorange")
plt.title("Advanced 4: effective decay update")
plt.ylabel("||ηλw||")
plt.show()

▶ What you'll see: the norm of the regularization movement falls with the learning rate.

👀 Takeaway: scheduled learning rates also schedule any update term multiplied by the rate.

### Advanced 5 — Choose a schedule with validation loss

**Goal.** Compare schedules on noisy train and validation curves, because the best schedule is the one that generalizes rather than merely moving fastest.

In [ ]:
steps_a5 = 50
rng_a5 = np.random.default_rng(20)
base_train_a5 = np.exp(-0.10 * np.arange(steps_a5))
base_val_a5 = 0.28 + 0.65 * np.exp(-0.08 * np.arange(steps_a5))
schedules_a5 = {
    "constant": np.full(steps_a5, 0.08),
    "step": 0.14 * 0.5 ** (np.arange(steps_a5) // 15),
    "cosine": 0.01 + 0.5 * (0.14 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_a5) / (steps_a5 - 1))),
}
print("schedule names:", list(schedules_a5.keys()))
assert len(schedules_a5) == 3

▶ What you'll see: three candidate schedules will be evaluated.

In [ ]:
val_curves_a5 = {}
for name_a5, sched_a5 in schedules_a5.items():
    smooth_bonus_a5 = 0.08 * (sched_a5 / sched_a5.max())
    noise_a5 = rng_a5.normal(0, 0.01, size=steps_a5)
    val_curves_a5[name_a5] = base_val_a5 + smooth_bonus_a5 + noise_a5
final_vals_a5 = {name_a5: float(curve_a5[-1]) for name_a5, curve_a5 in val_curves_a5.items()}
best_name_a5 = min(final_vals_a5, key=final_vals_a5.get)
print("final validation losses:", {k: round(v, 3) for k, v in final_vals_a5.items()})
print("best schedule:", best_name_a5)
assert best_name_a5 in schedules_a5

▶ What you'll see: each schedule gets a final validation loss and one is selected.

In [ ]:
plt.figure(figsize=(5, 3))
for name_a5, curve_a5 in val_curves_a5.items():
    plt.plot(curve_a5, label=name_a5)
plt.title("Advanced 5: validation curves by schedule")
plt.xlabel("step")
plt.ylabel("validation loss")
plt.legend()
plt.show()

▶ What you'll see: the curves differ slightly; the lowest final validation curve wins.

👀 Takeaway: schedule choice is a validation decision, not just a prettier learning-rate plot.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Schedules spend learning rate where it helps: gentle warmup, decisive movement, then careful settling.

The optimizer uses the same gradient formula, but the step size changes over time. Step decay, cosine decay, warmup, and one-cycle policies allocate movement differently. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, standardize, predict, and return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def one_hot(y, k):
    out = np.zeros((len(y), k))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def random_relu_features(X, seed=0, width=24):
    rng = np.random.default_rng(seed + X.shape[1])
    W = rng.normal(0.0, 1.0 / np.sqrt(max(1, X.shape[1])), size=(X.shape[1], width))
    b = rng.normal(0.0, 0.15, size=width)
    H = np.maximum(0.0, X @ W + b)
    pair = X[:, :1] * X[:, 1:2] if X.shape[1] >= 2 else X
    return np.hstack([X, X * X, pair, H])


def batch_norm_fit(H, eps=1e-5):
    mu = H.mean(axis=0, keepdims=True)
    var = H.var(axis=0, keepdims=True)
    Z = (H - mu) / np.sqrt(var + eps)
    return Z, (mu, var, eps)


def batch_norm_apply(H, params):
    mu, var, eps = params
    return (H - mu) / np.sqrt(var + eps)


def layer_norm(H, eps=1e-5):
    mu = H.mean(axis=1, keepdims=True)
    var = H.var(axis=1, keepdims=True)
    return (H - mu) / np.sqrt(var + eps)


def group_norm(H, groups=4, eps=1e-5):
    usable = (H.shape[1] // groups) * groups
    head = H[:, :usable].reshape(H.shape[0], groups, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def instance_norm(H, eps=1e-5):
    usable = (H.shape[1] // 8) * 8
    head = H[:, :usable].reshape(H.shape[0], 8, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def deep_random_features(X, depth=4, scale=1.0, residual=False, seed=0):
    H = random_relu_features(X, seed=seed, width=20)
    rng = np.random.default_rng(seed + 100 + H.shape[1])
    for _ in range(depth):
        W = rng.normal(0.0, scale / np.sqrt(H.shape[1]), size=(H.shape[1], H.shape[1]))
        F = np.maximum(0.0, H @ W)
        if residual:
            H = H + 0.35 * F
        else:
            H = F
    return H


def transform_pair(x_tr, x_te, mode="plain", seed=0, scale=1.0, residual=False):
    Htr = random_relu_features(x_tr, seed=seed)
    Hte = random_relu_features(x_te, seed=seed)
    if mode == "batchnorm":
        Htr, params = batch_norm_fit(Htr)
        Hte = batch_norm_apply(Hte, params)
    if mode == "test_batchnorm_wrong":
        Htr, params = batch_norm_fit(Htr)
        Hte, _ = batch_norm_fit(Hte)
    if mode == "layernorm":
        Htr = layer_norm(Htr)
        Hte = layer_norm(Hte)
    if mode == "groupnorm":
        Htr = group_norm(Htr)
        Hte = group_norm(Hte)
    if mode == "instancenorm":
        Htr = instance_norm(Htr)
        Hte = instance_norm(Hte)
    if mode == "deep":
        Htr = deep_random_features(x_tr, depth=5, scale=scale, residual=residual, seed=seed)
        Hte = deep_random_features(x_te, depth=5, scale=scale, residual=residual, seed=seed)
    return Htr, Hte


def train_softmax_classifier(x_tr, y_tr, x_te, epsilon=0.0, epochs=40, lr=0.2, clip=None, schedule="constant", transform="plain", seed=0, scale=1.0, residual=False):
    Htr, Hte = transform_pair(x_tr, x_te, mode=transform, seed=seed, scale=scale, residual=residual)
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 700)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    targets = (1.0 - epsilon) * Y + epsilon / k
    rng = np.random.default_rng(seed + 10)
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    losses = []
    grad_norms = []
    for epoch in range(epochs):
        eta = lr_value(schedule, epoch, epochs, lr)
        P = softmax(Htr @ W + b)
        loss = -np.mean(np.sum(targets * np.log(P + 1e-12), axis=1))
        G = (P - targets) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        norm = float(np.sqrt(np.sum(dW * dW) + np.sum(db * db)))
        if clip is not None:
            factor = min(1.0, clip / (norm + 1e-12))
            dW = dW * factor
            db = db * factor
        W = W - eta * dW
        b = b - eta * db
        losses.append(float(loss))
        grad_norms.append(norm)
    preds = np.argmax(Hte @ W + b, axis=1)
    return preds, losses, grad_norms


def lr_value(schedule, epoch, epochs, base):
    if schedule == "constant":
        return base
    if schedule == "step":
        return base if epoch < epochs // 2 else base * 0.2
    if schedule == "cosine":
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * epoch / max(1, epochs - 1)))
    if schedule == "warmup_cosine":
        warm = max(2, epochs // 5)
        if epoch < warm:
            return base * (epoch + 1) / warm
        span = max(1, epochs - warm - 1)
        t = epoch - warm
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * t / span))
    if schedule == "onecycle":
        half = max(1, epochs // 2)
        if epoch < half:
            return base * (0.2 + 1.8 * epoch / half)
        return base * (2.0 - 1.8 * (epoch - half) / max(1, epochs - half))
    return base


def component_accuracy(name, X, y, **kwargs):
    def build(x_tr, y_tr, x_te):
        preds, _, _ = train_softmax_classifier(x_tr, y_tr, x_te, **kwargs)
        return preds
    return clf_accuracy(build, X, y)


def fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.3, seed=0):
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 701)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    rng = np.random.default_rng(seed + Htr.shape[1])
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    for epoch in range(epochs):
        P = softmax(Htr @ W + b)
        G = (P - Y) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        W = W - lr * dW
        b = b - lr * db
    return np.argmax(Hte @ W + b, axis=1)


def logistic_accuracy_for_features(X, y, mode="plain", scale=1.0, residual=False, seed=0):
    def build(x_tr, y_tr, x_te):
        Htr, Hte = transform_pair(x_tr, x_te, mode=mode, seed=seed, scale=scale, residual=residual)
        return fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.35, seed=seed)
    return clf_accuracy(build, X, y)


def ladder_preview(rungs):
    for name, X, y in rungs:
        classes = np.unique(y)
        print(f"{name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")
    print("First D1 sample:", rungs[0][1][0].tolist(), "label=", int(rungs[0][2][0]))


def print_metric_table(rows, header="rung metric"):
    print(header)
    for name, metric in rows:
        print(f"{name:36s} {metric:.3f}")


def split_for_demo(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def plot_ladder_results(rungs, metrics, title, artifact_fn=None):
    fig, axes = plt.subplots(1, 5, figsize=(16, 3))
    for ax, (name, X, y) in zip(axes, rungs):
        if artifact_fn is None:
            if X.shape[1] == 64:
                ax.imshow(X[0].reshape(8, 8), cmap="gray")
            else:
                ax.scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=12)
        else:
            artifact_fn(ax, name, X, y)
        ax.set_title(name.split()[0])
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title + " artifacts")
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.plot(range(1, 6), metrics, marker="o")
    plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylim(0.0, 1.05)
    plt.ylabel("held-out accuracy")
    plt.title(title + " summary")
    plt.grid(True, alpha=0.3)
    plt.show()

## The concept, built once

The lesson formula is $\eta_t=\eta_{min}+\frac12(\eta_{max}-\eta_{min})(1+\cos(\pi t/T))$. For $\eta_{min}=0.01$, $\eta_{max}=0.10$, $T=4$, and $t=0..4$, the cosine schedule is hand-checkable as $[0.10000,0.08682,0.05500,0.02318,0.01000]$.

In [ ]:
eta_min = 0.01
eta_max = 0.10
T = 4
trace = []
for t in range(5):
    eta_t = eta_min + 0.5 * (eta_max - eta_min) * (1.0 + math.cos(math.pi * t / T))
    trace.append(eta_t)
expected = np.array([0.1, 0.086819805, 0.055, 0.023180195, 0.01])
print("cosine trace:", [round(v, 5) for v in trace])
assert np.allclose(trace, expected, atol=1e-8)

This helper is the reusable method for the rest of the notebook. It keeps the model and ladder fixed, then varies only this topic's component.

In [ ]:
print('Reusable component method is available in the setup cell and verified above.')

## The dataset ladder

We use the shared F5 classification ladder: D1 XOR, D2 blobs, D3 noisy moons, D4 real sklearn digits, and D5 noisy digits. The same accuracy wrapper and model family run on every rung.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1–D5

The table reports one held-out accuracy per rung while the component-specific sweep is printed for auditability.

In [ ]:
rungs = clf_digits_ladder()
rows = []
for rung_id, (name, X, y) in enumerate(rungs):
    vals = []
    for schedule in ["step", "cosine", "warmup_cosine", "onecycle"]:
        acc = component_accuracy(name, X, y, epochs=40, lr=0.35, schedule=schedule, seed=80 + rung_id)
        vals.append((schedule, acc))
    chosen = [acc for schedule, acc in vals if schedule == "warmup_cosine"][0]
    rows.append((name, chosen))
    print(name, [(schedule, round(acc, 3)) for schedule, acc in vals])
metrics = [metric for _, metric in rows]
print_metric_table(rows, "warmup+cosine accuracy")

## Results visualization

The closing figure has two parts: a small multiple showing each rung's data artifact and a summary curve of the selected metric from D1 to D5.

In [ ]:
plot_ladder_results(rungs, metrics, '6.20 Learning-rate schedules')

## Pitfall on D5

Aggressive one-cycle peaks can spike D5 loss. Bound the peak rate and use warmup/cosine when early steps are unstable.

In [ ]:
name, X, y = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_for_demo(X, y)
preds_one, losses_one, _ = train_softmax_classifier(x_tr, y_tr, x_te, epochs=50, lr=0.7, schedule="onecycle", seed=111)
preds_warm, losses_warm, _ = train_softmax_classifier(x_tr, y_tr, x_te, epochs=50, lr=0.35, schedule="warmup_cosine", seed=111)
print("D5 one-cycle max loss:", round(max(losses_one), 3), "final:", round(losses_one[-1], 3))
print("D5 warmup-cosine max loss:", round(max(losses_warm), 3), "final:", round(losses_warm[-1], 3))
print("Fix: lower eta_max or add warmup before the high-rate phase.")

## Evaluate it

- Metric: held-out accuracy from `clf_accuracy`; compare to a no-skill majority-class or plain-feature baseline.
- Sanity check: D1 should be inspectable and every probability target should sum to one when probabilities are used.
- Ablation: turn this topic's component off and verify the metric or diagnostic changes.
- Failure signal: unstable loss, axis mismatch, train/eval leakage, or D5 improvement without a matching diagnostic.

## Practice

1. Change one component value and rerun the D1 assertion plus the D1–D5 table.

2. Add a majority-class baseline to the summary curve.

3. On D5, print one extra diagnostic that would catch the named pitfall.